# NB03: CAZy Density and Pangenome Openness (H2)

Test H2: CAZyme gene density is positively correlated with pangenome openness (cloud gene fraction).
Control for genome_count (sampling effort) and genome size via partial correlation.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = '../data'
FIG_DIR = '../figures'

## 1. Merge species-level CAZy means with pangenome stats

In [ ]:
wide = pd.read_csv(f'{DATA_DIR}/genome_cazy_profiles.tsv', sep='\t')
pangenome = pd.read_csv(f'{DATA_DIR}/pangenome_stats.tsv', sep='\t')

cazy_classes = ['AA', 'CB', 'CE', 'GH', 'GT', 'PL']
species_cazy = wide.groupby('species_id').agg(
    biome_id=('biome_id', 'first'),
    gtdb_phylum=('gtdb_phylum', 'first'),
    length=('length', 'mean'),
    genome_type=('genome_type', 'first'),
    **{c: (c, 'mean') for c in cazy_classes},
    total_cazy=('total_cazy', 'mean'),
    cazy_density=('cazy_density', 'mean')
).reset_index()

merged = species_cazy.merge(pangenome, on='species_id', how='inner')
print(f'Species with both CAZy and pangenome data: {len(merged):,}')
merged.to_csv(f'{DATA_DIR}/species_cazy_pangenome.tsv', sep='\t', index=False)

## 2. Raw correlation: total CAZy vs cloud fraction

In [ ]:
rho, p = stats.spearmanr(merged['total_cazy'], merged['cloud_fraction'])
print(f'Spearman rho = {rho:.3f}, p = {p:.2e}')

rho_gc, p_gc = stats.spearmanr(merged['genome_count'], merged['cloud_fraction'])
print(f'genome_count vs cloud_fraction: rho = {rho_gc:.3f}, p = {p_gc:.2e}')
print('WARNING: genome_count dominates cloud_fraction — major confound for H2')

## 3. Partial correlation controlling for genome size and genome count

In [ ]:
from numpy.linalg import lstsq

def partial_spearman(x, y, covariates):
    """Spearman partial correlation controlling for covariates."""
    x_rank = stats.rankdata(x)
    y_rank = stats.rankdata(y)
    cov_ranks = np.column_stack([stats.rankdata(c) for c in covariates.T])
    cov_ranks = np.column_stack([cov_ranks, np.ones(len(x_rank))])
    x_resid = x_rank - cov_ranks @ lstsq(cov_ranks, x_rank, rcond=None)[0]
    y_resid = y_rank - cov_ranks @ lstsq(cov_ranks, y_rank, rcond=None)[0]
    return stats.pearsonr(x_resid, y_resid)

covariates = merged[['length', 'genome_count']].values
rho_partial, p_partial = partial_spearman(
    merged['total_cazy'].values,
    merged['cloud_fraction'].values,
    covariates
)
print(f'Partial Spearman (controlling genome_size + genome_count): rho = {rho_partial:.3f}, p = {p_partial:.2e}')

## 4. Scatter plot: CAZy vs openness

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(merged['cloud_fraction'], merged['total_cazy'],
                alpha=0.1, s=5, c='steelblue')
axes[0].set_xlabel('Cloud Gene Fraction')
axes[0].set_ylabel('Total CAZy Genes (species mean)')
axes[0].set_title(f'Raw: ρ = {rho:.3f}')

top_biomes = merged['biome_id'].value_counts().head(6).index
colors = plt.cm.Set2(np.linspace(0, 1, len(top_biomes)))
for biome, color in zip(top_biomes, colors):
    sub = merged[merged['biome_id'] == biome]
    axes[1].scatter(sub['cloud_fraction'], sub['total_cazy'],
                    alpha=0.3, s=5, label=biome, color=color)
axes[1].set_xlabel('Cloud Gene Fraction')
axes[1].set_ylabel('Total CAZy Genes (species mean)')
axes[1].set_title('By Biome')
axes[1].legend(markerscale=3, fontsize=8)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/cazy_vs_openness.png', dpi=150)
plt.show()

## 5. Within-phylum correlations

In [ ]:
top_phyla = merged['gtdb_phylum'].value_counts().head(10).index
phylum_corr = []

for phylum in top_phyla:
    sub = merged[merged['gtdb_phylum'] == phylum]
    rho, p = stats.spearmanr(sub['total_cazy'], sub['cloud_fraction'])
    phylum_corr.append({'phylum': phylum, 'n': len(sub), 'rho': rho, 'p': p})

corr_df = pd.DataFrame(phylum_corr)
corr_df['p_adj'] = multipletests(corr_df['p'], method='fdr_bh')[1]
print(corr_df.to_string(index=False))
corr_df.to_csv(f'{DATA_DIR}/phylum_cazy_openness_corr.tsv', sep='\t', index=False)

## Summary

- H2 weakly supported: raw Spearman ρ = 0.153, partial ρ = 0.101
- genome_count → cloud_fraction confound is severe (ρ = 0.913)
- Signal driven by Bacillota_A (ρ = 0.288), Bacteroidota (ρ = 0.147), Firmicutes_A (ρ = 0.105)
- 7/10 phyla show no significant correlation